# CPE Perception Pipeline

In [ ]:
%pip install -q google-cloud-storage opencv-python ultralytics lapx numpy matplotlib torch
print("Dependencies installed")

Dependencies installed


## Mount Google Drive


In [ ]:
import os
from pathlib import Path
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

GOOGLE_DRIVE_ROOT = Path("/content/drive/My Drive")
DRIVE_DATA_ROOT = GOOGLE_DRIVE_ROOT / "Capstone_Data_new"
DRIVE_VALID_STREAMS_FILE = DRIVE_DATA_ROOT / "valid_streams.json"
DRIVE_PROCESSED_STREAMS_FILE = DRIVE_DATA_ROOT / "processed_streams.json"
DRIVE_SANPO_JSON_PATH = DRIVE_DATA_ROOT / "Data_new"

def ensure_drive_dir(path: Path, label: str = "path") -> bool:
    """Best-effort Drive path check/create that avoids hard crashes on DriveFS latency."""
    try:
        if path.exists():
            return True
    except TimeoutError:
        print(f"[WARN] Drive path check timed out for {label}: {path}")
        return False

    try:
        path.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] Created missing Drive {label}: {path}")
        return True
    except TimeoutError:
        print(f"[WARN] Drive path create timed out for {label}: {path}")
        return False
    except Exception as exc:
        print(f"[WARN] Could not prepare Drive {label} at {path}: {exc}")
        return False

for label, required_path in (
    ("data root", DRIVE_DATA_ROOT),
    ("session output folder", DRIVE_SANPO_JSON_PATH),
):
    ensure_drive_dir(required_path, label=label)

print(f"Google Drive root detected at: {GOOGLE_DRIVE_ROOT}")
print(f"Data root path: {DRIVE_DATA_ROOT}")
print(f"Valid streams file: {DRIVE_VALID_STREAMS_FILE}")
print(f"Processed streams file: {DRIVE_PROCESSED_STREAMS_FILE}")
print(f"Session output path: {DRIVE_SANPO_JSON_PATH}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive root detected at: /content/drive/My Drive
Data root path: /content/drive/My Drive/Capstone_Data_new
Valid streams file: /content/drive/My Drive/Capstone_Data_new/valid_streams.json
Processed streams file: /content/drive/My Drive/Capstone_Data_new/processed_streams.json
Session output path: /content/drive/My Drive/Capstone_Data_new/Data_new


## Imports

In [ ]:
import cv2, gzip, json, os, csv, textwrap
import io
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional, Generator, Tuple, List
from collections import defaultdict, deque
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials

## Frame Dataclass

In [ ]:
@dataclass
class Frame:
    frame_id:   int
    timestamp:  float
    rgb:        np.ndarray
    depth:      np.ndarray
    session_id: str
    camera:     str
    view:       str
    fps:        int
    resolution: Tuple[int, int]

## SANPO Loader

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

FRAME_STRIDE = 3
MISSING_FRAME_STREAK_LIMIT = 40
FRAME_FETCH_TIMEOUT_S = 6
MAX_WORKERS = 3
VELOCITY_WINDOW = 5

def load_processed_streams_manifest():
    if DRIVE_PROCESSED_STREAMS_FILE.exists():
        try:
            with open(DRIVE_PROCESSED_STREAMS_FILE) as f:
                manifest = json.load(f)
                return manifest if isinstance(manifest, dict) else {}
        except (json.JSONDecodeError, OSError):
            return {}
    return {}


def save_processed_streams_manifest(manifest):
    if not ensure_drive_dir(DRIVE_PROCESSED_STREAMS_FILE.parent, label="data root"):
        print(f"[WARN] Skipping processed-stream manifest write because Drive folder is unavailable: {DRIVE_PROCESSED_STREAMS_FILE.parent}")
        return False
    with open(DRIVE_PROCESSED_STREAMS_FILE, "w") as f:
        json.dump(manifest, f, indent=2)
    return True


processed_cache = {}

class ScenarioFilter:
    def __init__(self, min_frames=150):
        self.min_frames = min_frames

    def filter_by_metadata(self, meta):
        video_meta = meta.get("session_video_metadata", {})

        motion = video_meta.get("ego_motion", "")
        motion_ok = motion in ["EGO_MOTION_WALKING", "EGO_MOTION_RUNNING"]

        envs = video_meta.get("environment_types", [])
        env_ok = any(e in envs for e in [
            "ENVIRONMENT_TYPE_URBAN",
            "ENVIRONMENT_TYPE_ROAD_JUNCTION",
            "ENVIRONMENT_TYPE_SIDEWALK"
        ])

        human = video_meta.get("human_traffic", "")
        traffic_ok = human != "HUMAN_TRAFFIC_NONE"

        return motion_ok and env_ok and traffic_ok

class SANPOLoader:
    def __init__(self, sanpo_root="gs://gresearch/sanpo_dataset/v0/sanpo-real",
                 camera="chest", view="left"):
        if "sanpo-real" not in sanpo_root:
            sanpo_root = "gs://gresearch/sanpo_dataset/v0/sanpo-real"

        self.sanpo_root = sanpo_root.rstrip("/")
        self.camera = camera
        self.view = view
        credentials = AnonymousCredentials()
        self.client = storage.Client(credentials=credentials, project='gresearch')
        parts = self.sanpo_root.split("/")
        self.bucket_name = parts[2]
        self.prefix = "/".join(parts[3:])
        self.bucket = self.client.bucket(self.bucket_name)
        self.session_cache = None

    def list_sessions(self):
        """Fetches ALL available sessions from the real dataset bucket."""
        if self.session_cache is not None:
            return self.session_cache

        prefix = f"{self.prefix}/"
        sessions = []
        print(f"Scanning {self.sanpo_root} for real-world sessions...")

        for page in self.client.list_blobs(
            self.bucket_name, prefix=prefix, delimiter="/"
        ).pages:
            for p in page.prefixes:
                session_id = p[len(prefix):].rstrip("/")
                if session_id and "synthetic" not in session_id.lower():
                    sessions.append(session_id)

        self.session_cache = sorted(sessions)
        return self.session_cache

    def get_session_metadata(self, session_id):
        path = f"{self.prefix}/{session_id}/description.json"
        blob = self.bucket.blob(path)
        try:
            return json.loads(blob.download_as_string(timeout=FRAME_FETCH_TIMEOUT_S))
        except:
            return {}

    def _get_rgb(self, session_id, frame_id):
        path = f"{self.prefix}/{session_id}/camera_{self.camera}/{self.view}/video_frames/{frame_id:06d}.png"
        try:
            blob = self.bucket.blob(path)
            arr  = np.frombuffer(blob.download_as_bytes(timeout=FRAME_FETCH_TIMEOUT_S), np.uint8)
            img  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except:
            return None

    def _get_depth(self, session_id, frame_id):
        TARGET_H, TARGET_W = 720, 1280
        zed_path = (f"{self.prefix}/{session_id}/camera_{self.camera}"
                    f"/{self.view}/zed_depth_maps/{frame_id:06d}.npz")
        try:
            blob  = self.bucket.blob(zed_path)
            raw   = blob.download_as_bytes(timeout=FRAME_FETCH_TIMEOUT_S)
            depth = np.load(io.BytesIO(raw))["arr_0"].astype(np.float32)
            depth[~np.isfinite(depth)] = 0
            depth[depth > 100] = 0
            if depth.shape != (TARGET_H, TARGET_W):
                depth = cv2.resize(depth, (TARGET_W, TARGET_H), interpolation=cv2.INTER_NEAREST)
            return depth
        except:
            pass

        cre_path = (f"{self.prefix}/{session_id}/camera_{self.camera}"
                    f"/{self.view}/depth_maps/{frame_id:06d}.float16.gz")
        try:
            blob  = self.bucket.blob(cre_path)
            raw   = gzip.decompress(blob.download_as_bytes(timeout=FRAME_FETCH_TIMEOUT_S))
            depth = np.frombuffer(raw, dtype=np.float16)
            expected = TARGET_H * TARGET_W
            if len(depth) > expected: depth = depth[:expected]
            if len(depth) != expected: return None
            depth = depth.reshape(TARGET_H, TARGET_W).astype(np.float32)
            depth[depth > 100] = 0
            return depth
        except:
            return None

    def _load_frame_pair(self, session_id, frame_id, fps):
        rgb = self._get_rgb(session_id, frame_id)
        depth = self._get_depth(session_id, frame_id)
        if rgb is None or depth is None:
            return None
        return Frame(
            frame_id=frame_id, timestamp=frame_id / fps,
            rgb=rgb, depth=depth, session_id=session_id,
            camera=self.camera, view=self.view, fps=fps,
            resolution=(rgb.shape[1], rgb.shape[0])
        )

    def iter_frames(self, session_id, max_frames=5, workers=MAX_WORKERS, start_frame=0, missing_streak_limit=MISSING_FRAME_STREAK_LIMIT):
        meta = self.get_session_metadata(session_id)
        fps = meta.get("fps", 15)
        missing_streak = 0

        with ThreadPoolExecutor(max_workers=workers) as executor:
            batch_span = FRAME_STRIDE * max(1, workers)
            for batch_start in range(start_frame, max_frames, batch_span):
                batch_end = min(max_frames, batch_start + batch_span)
                frame_ids = list(range(batch_start, batch_end, FRAME_STRIDE))
                futures = [executor.submit(self._load_frame_pair, session_id, fid, fps)
                           for fid in frame_ids]

                for fid, future in zip(frame_ids, futures):
                    try:
                        frame = future.result(timeout=FRAME_FETCH_TIMEOUT_S)
                    except FuturesTimeoutError:
                        frame = None

                    if frame is None:
                        missing_streak += 1
                        if missing_streak >= missing_streak_limit:
                            print(f"[INFO] Stopping {session_id}: reached {missing_streak_limit} consecutive missing/timeout frames.")
                            return
                        continue

                    missing_streak = 0
                    yield frame

print("SANPOLoader updated for Drive-backed manifest and resume support")

SANPOLoader updated for Drive-backed manifest and resume support


## Physics

In [ ]:
def compute_bearing(cx_px, frame_width, hfov_deg=70.0):
    normalised = (cx_px - frame_width / 2) / (frame_width / 2)
    return normalised * (hfov_deg / 2)

def compute_velocity(depth_history, fps):
    if len(depth_history) < 2:
        return 0.0
    (f0, d0) = depth_history[0]
    (f1, d1) = depth_history[-1]
    dt = (f1 - f0) / fps
    if dt <= 0:
        return 0.0
    return max(0.0, (d0 - d1) / dt)

def bearing_label(deg):
    if deg < -30: return "far-left"
    if deg < -10: return "left"
    if deg < 10:  return "ahead"
    if deg < 30:  return "right"
    return "far-right"

print("Physics ready")

Physics ready


## Depth Utilities

In [ ]:
MIN_DEPTH_M = 0.3
MAX_DEPTH_M = 30.0

def median_depth_in_box(depth_map, x1, y1, x2, y2,
                         center_frac=0.2, exclude_boxes=None):
    """
    Sample depth from the central patch of a bounding box.

    exclude_boxes: list of (x1,y1,x2,y2) boxes to mask out before sampling.
    Used to prevent a background object from reading depth through a
    foreground object that overlaps it in image space.
    """
    h, w = depth_map.shape[:2]

    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2
    bw = max(1, int((x2 - x1) * center_frac / 2))
    bh = max(1, int((y2 - y1) * center_frac / 2))

    rx1, rx2 = max(0, cx - bw), min(w, cx + bw)
    ry1, ry2 = max(0, cy - bh), min(h, cy + bh)
    if rx1 >= rx2 or ry1 >= ry2:
        return None

    # Build a mask for the sampling patch — True = pixel is usable
    patch_h = ry2 - ry1
    patch_w = rx2 - rx1
    mask = np.ones((patch_h, patch_w), dtype=bool)

    # Mask out pixels that belong to any foreground (occluding) box
    if exclude_boxes:
        for (ex1, ey1, ex2, ey2) in exclude_boxes:
            # Intersect the exclude box with our sampling patch in patch coords
            mx1 = max(0, ex1 - rx1)
            my1 = max(0, ey1 - ry1)
            mx2 = min(patch_w, ex2 - rx1)
            my2 = min(patch_h, ey2 - ry1)
            if mx1 < mx2 and my1 < my2:
                mask[my1:my2, mx1:mx2] = False

    roi   = depth_map[ry1:ry2, rx1:rx2]
    valid = roi[(roi > MIN_DEPTH_M) & (roi < MAX_DEPTH_M) & mask]

    # ZED sparse fallback — expand patch if no readings found
    if valid.size == 0:
        bw2, bh2 = bw * 3, bh * 3
        rx1, rx2 = max(0, cx - bw2), min(w, cx + bw2)
        ry1, ry2 = max(0, cy - bh2), min(h, cy + bh2)
        patch_h = ry2 - ry1
        patch_w = rx2 - rx1
        mask2 = np.ones((patch_h, patch_w), dtype=bool)
        if exclude_boxes:
            for (ex1, ey1, ex2, ey2) in exclude_boxes:
                mx1 = max(0, ex1 - rx1); my1 = max(0, ey1 - ry1)
                mx2 = min(patch_w, ex2 - rx1); my2 = min(patch_h, ey2 - ry1)
                if mx1 < mx2 and my1 < my2:
                    mask2[my1:my2, mx1:mx2] = False
        roi   = depth_map[ry1:ry2, rx1:rx2]
        valid = roi[(roi > MIN_DEPTH_M) & (roi < MAX_DEPTH_M) & mask2]

    if valid.size == 0:
        return None
    return float(np.median(valid))

print("Depth utils ready")


Depth utils ready


## YOLO Tracker

In [ ]:
from ultralytics import YOLO
import torch

# Keep the requested model exactly as-is
YOLO_MODEL_PATH = "yolo26n.pt"


def _resolve_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return "mps"
    return "cpu"


DEVICE = _resolve_device()
print(f"Initializing YOLO with {YOLO_MODEL_PATH} on {DEVICE.upper()}...")
try:
    tracker_model = YOLO(YOLO_MODEL_PATH)
    tracker_model.to(DEVICE)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

ALLOWED_CLASSES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "dog", "cat", "traffic light", "stop sign", "umbrella",
    "backpack", "suitcase", "unlabeled_obstacle"
}


class YoloTracker:
    def __init__(self, model_instance=tracker_model, device=DEVICE):
        self.model = model_instance
        self.device = device

    def track(self, rgb_frame):
        return self.model.track(rgb_frame, persist=True, verbose=False, device=self.device)[0]

Initializing YOLO with yolo26n.pt on CUDA...
Model loaded successfully.


## Stream Pipeline

In [ ]:
import time

def run_stream_session(session_id, frame_generator, fps, out_jsonl_path, resume_from_frame=0, checkpoint_manifest=None):
    tracker = YoloTracker()
    print(f'  -> Starting Perception Pipeline on {tracker.device.upper()}...')
    track_history = defaultdict(lambda: deque(maxlen=VELOCITY_WINDOW))
    pending_resume_frame = resume_from_frame
    processed_frames = 0
    progress_every = 25
    start_time = time.perf_counter()

    try:
        with open(out_jsonl_path, 'a') as f_out:
            for frame_data in frame_generator:
                fid = frame_data.frame_id
                pending_resume_frame = fid
                rgb = frame_data.rgb
                depth = frame_data.depth
                h, w = rgb.shape[:2]

                # 1. Get RGB detections (YOLO + ByteTrack)
                res_rgb = tracker.track(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

                # 2. Build frame object list
                objects_in_frame = []
                if res_rgb.boxes is not None and len(res_rgb.boxes) > 0:
                    boxes = res_rgb.boxes.xyxy.cpu().numpy()
                    classes = res_rgb.boxes.cls.cpu().numpy().astype(int)
                    confs = res_rgb.boxes.conf.cpu().numpy()
                    ids = res_rgb.boxes.id.cpu().numpy().astype(int) if res_rgb.boxes.id is not None else [None] * len(boxes)

                    for i, (box, tid, cls_idx, conf) in enumerate(zip(boxes, ids, classes, confs)):
                        # Keep only navigation-relevant classes
                        class_name = res_rgb.names.get(int(cls_idx), 'unknown')
                        if class_name not in ALLOWED_CLASSES:
                            continue

                        conf_val = float(conf) if conf is not None else 0.0
                        if conf_val < 0.25:
                            continue

                        x1, y1, x2, y2 = map(int, box)

                        dist = median_depth_in_box(depth, x1, y1, x2, y2)
                        velocity = 0.0

                        # ByteTrack IDs are missing on some frames; do not drop detections in that case.
                        track_id = str(int(tid)) if tid is not None else None
                        if dist is not None and track_id is not None:
                            track_history[track_id].append((fid, dist))
                            velocity = compute_velocity(list(track_history[track_id]), fps)

                        bearing_deg = compute_bearing((x1 + x2) / 2, w)
                        position_label = bearing_label(bearing_deg)

                        det_obj = {
                            'object_id': f'Object_{i+1:02d}',
                            'track_id': track_id,
                            'class': class_name,
                            'distance_m': round(dist, 2) if dist is not None else None,
                            'velocity_ms': round(velocity, 3),
                            'bearing_deg': round(bearing_deg, 2),
                            'position_label': position_label,
                            'bbox_xyxy': [x1, y1, x2, y2],
                            'confidence': round(conf_val, 2),
                        }
                        objects_in_frame.append(det_obj)

                frame_output = {
                    'schema_version': '1.0',
                    'session_id': session_id,
                    'frame_id': fid,
                    'timestamp_s': round(fid / fps, 2),
                    'fps': fps,
                    'objects': objects_in_frame,
                }
                f_out.write(json.dumps(frame_output) + '\n')

                processed_frames += 1
                if processed_frames == 1 or processed_frames % progress_every == 0:
                    elapsed = max(time.perf_counter() - start_time, 1e-6)
                    fps_eff = processed_frames / elapsed
                    print(
                        f"[PROGRESS] {session_id}: frames={processed_frames}, last_frame={fid}, "
                        f"objects={len(objects_in_frame)}, speed={fps_eff:.2f} frames/s"
                    )

                pending_resume_frame = fid + FRAME_STRIDE

        print(f'\n✅ Session Saved: Output for {processed_frames} frames written to {out_jsonl_path}')
        return True

    except KeyboardInterrupt:
        if checkpoint_manifest is not None:
            checkpoint_manifest[session_id] = {
                'completed': False,
                'last_frame_id': pending_resume_frame,
            }
            save_processed_streams_manifest(checkpoint_manifest)
            print(f"[CHECKPOINT] Saved interrupted frame checkpoint for {session_id} at frame {pending_resume_frame}")
        raise
    except Exception as exc:
        print(f"[ERROR] Session {session_id} failed: {exc}")
        return False

# Clear the valid session ids



In [ ]:
VALID_SESSIONS_FILE = DRIVE_VALID_STREAMS_FILE

print("🧹 Clearing valid sessions file from Drive...")
try:
    if VALID_SESSIONS_FILE.exists():
        VALID_SESSIONS_FILE.unlink()
        print(f"⌒ File '{VALID_SESSIONS_FILE}' deleted from Drive.")
    else:
        print(f"'{VALID_SESSIONS_FILE}' does not exist. Nothing to clear.")
except TimeoutError:
    print(f"[WARN] Drive timed out while checking/deleting: {VALID_SESSIONS_FILE}")

🧹 Clearing valid sessions file from Drive...
⌒ File '/Users/ksrikrishnareddy/Library/CloudStorage/GoogleDrive-srikrishnareddyk2005@gmail.com/My Drive/Capstone_Data/valid_streams.json' deleted from Drive.


## Initialise Loader & List Sessions

In [ ]:
loader = SANPOLoader()
scenario_filter = ScenarioFilter()

# Fetch ALL session IDs from the entire real-world bucket
all_sessions = loader.list_sessions()

CAMERA_PRIORITY = ["head", "chest"]
VIEWS = ["left", "right"]
# We will find ALL valid sessions but only run a few later
MAX_VALID_STREAMS = 9999

valid_streams = []

def stream_exists(session_id, camera, view):
    prefix = f"{loader.prefix}/{session_id}/camera_{camera}/{view}/video_frames/000000.png"
    return loader.bucket.blob(prefix).exists()

print(f"Scanning through {len(all_sessions)} sessions to identify all that meet criteria...")
for s in all_sessions:
    try:
        meta = loader.get_session_metadata(s)
        if not scenario_filter.filter_by_metadata(meta):
            continue

        # Search for available camera/view stream
        stream_found = False
        for camera in CAMERA_PRIORITY:
            for view in VIEWS:
                if stream_exists(s, camera, view):
                    valid_streams.append({"session_id": s, "camera": camera, "view": view})
                    stream_found = True
                    break
            if stream_found: break
    except:
        continue

print(f"\n✓ Total sessions in SANPO-Real bucket: {len(all_sessions)}")
print(f"✓ Total sessions meeting criteria discovered: {len(valid_streams)}")

Scanning gs://gresearch/sanpo_dataset/v0/sanpo-real for real-world sessions...
Scanning through 702 sessions to identify all that meet criteria...

✓ Total sessions in SANPO-Real bucket: 702
✓ Total sessions meeting criteria discovered: 462


# Save valid session ids

In [ ]:
import json

VALID_SESSIONS_FILE = DRIVE_VALID_STREAMS_FILE

if ensure_drive_dir(VALID_SESSIONS_FILE.parent, label="data root"):
    with open(VALID_SESSIONS_FILE, "w") as f:
        json.dump(valid_streams, f, indent=2)
    print(f"Valid sessions saved to {VALID_SESSIONS_FILE}")
else:
    print("[WARN] Could not save valid sessions because Drive data root is unavailable right now.")

Valid sessions saved to /Users/ksrikrishnareddy/Library/CloudStorage/GoogleDrive-srikrishnareddyk2005@gmail.com/My Drive/Capstone_Data/valid_streams.json


## ▶️ Run Pipeline
Adjust `MAX_FRAMES` and `STREAMS_TO_RUN` as needed.

In [ ]:
import json
from pathlib import Path

def load_valid_streams_from_drive(valid_streams_file: Path):
    if not valid_streams_file.exists():
        print(f"[STOPPED] Valid streams file not found: {valid_streams_file}")
        return []
    try:
        with open(valid_streams_file, "r") as f:
            data = json.load(f)
    except Exception as exc:
        print(f"[STOPPED] Could not read valid streams file: {exc}")
        return []

    if not isinstance(data, list):
        print("[STOPPED] valid_streams.json must contain a list of objects.")
        return []

    cleaned = []
    for row in data:
        if isinstance(row, dict) and row.get("session_id") and row.get("camera") and row.get("view"):
            cleaned.append({
                "session_id": row["session_id"],
                "camera": row["camera"],
                "view": row["view"],
            })
    return cleaned

bootstrap_required = [
    "ensure_drive_dir",
    "DRIVE_DATA_ROOT",
    "DRIVE_SANPO_JSON_PATH",
    "DRIVE_PROCESSED_STREAMS_FILE",
    "DRIVE_VALID_STREAMS_FILE",
]
missing_bootstrap = [name for name in bootstrap_required if name not in globals()]
if missing_bootstrap:
    print(f"[STOPPED] Missing bootstrap definitions: {', '.join(missing_bootstrap)}")
    print("[ACTION] Run the Drive setup cell first, then rerun this cell.")
    STREAMS_TO_RUN = []
else:
    manifest_writable = ensure_drive_dir(DRIVE_DATA_ROOT, label="data root")
    output_writable = ensure_drive_dir(DRIVE_SANPO_JSON_PATH, label="session output folder")

    required_symbols = [
        "YoloTracker", "Frame", "compute_bearing", "compute_velocity",
        "median_depth_in_box", "SANPOLoader", "bearing_label"
    ]
    missing_symbols = [name for name in required_symbols if name not in globals()]

    if missing_symbols:
        print(f"[STOPPED] Missing required definitions in current kernel: {', '.join(missing_symbols)}")
        print("[ACTION] Run model/pipeline cells (Imports through Stream Pipeline), then rerun.")
        STREAMS_TO_RUN = []
    else:
        loader = SANPOLoader()
        loader.sanpo_root = "gs://gresearch/sanpo_dataset/v0/sanpo-real"

        if manifest_writable and DRIVE_PROCESSED_STREAMS_FILE.exists():
            processed_manifest = load_processed_streams_manifest()
            print(f"Loaded {len(processed_manifest)} processed-stream manifest entries from Drive.")
        else:
            processed_manifest = {}
            if manifest_writable:
                print("No previous processed-stream manifest found on Drive. Starting fresh.")
            else:
                print("[WARN] Drive data root unavailable; manifest resume is disabled for this run.")

        completed_sessions = {
            session_id for session_id, record in processed_manifest.items()
            if isinstance(record, dict) and record.get("completed")
        }
        checkpoint_sessions = {
            session_id: record for session_id, record in processed_manifest.items()
            if isinstance(record, dict) and not record.get("completed") and record.get("last_frame_id") is not None
        }

        existing_output_sessions = {path.stem for path in DRIVE_SANPO_JSON_PATH.glob("*.jsonl")}
        for session_id in existing_output_sessions:
            if session_id not in completed_sessions and session_id not in checkpoint_sessions:
                completed_sessions.add(session_id)
                processed_manifest[session_id] = {"completed": True, "last_frame_id": None}

        if manifest_writable:
            save_processed_streams_manifest(processed_manifest)

        valid_streams = load_valid_streams_from_drive(DRIVE_VALID_STREAMS_FILE)
        all_streams_to_consider = [s for s in valid_streams if s["session_id"] not in completed_sessions]

        STREAMS_TO_RUN = all_streams_to_consider[:20]

        MAX_FRAMES = 9999
        total_sessions_processed = 0

        print(f"🚀 Starting pipeline for {len(STREAMS_TO_RUN)} unprocessed valid sessions (max 20). Output to {DRIVE_SANPO_JSON_PATH}")

        for stream_info in STREAMS_TO_RUN:
            session_id = stream_info["session_id"]
            session_jsonl = DRIVE_SANPO_JSON_PATH / f"{session_id}.jsonl"
            start_frame = int(checkpoint_sessions.get(session_id, {}).get("last_frame_id") or 0)

            loader.camera = stream_info["camera"]
            loader.view = stream_info["view"]

            # Use session metadata FPS when available so timestamps/velocity are correct.
            session_meta = loader.get_session_metadata(session_id)
            session_fps = float(session_meta.get("fps") or 15.0)

            try:
                frame_gen = loader.iter_frames(
                    session_id, max_frames=MAX_FRAMES, workers=MAX_WORKERS, start_frame=start_frame
                )
                success = run_stream_session(
                    session_id=session_id,
                    frame_generator=frame_gen,
                    fps=session_fps,
                    out_jsonl_path=session_jsonl,
                    resume_from_frame=start_frame,
                    checkpoint_manifest=processed_manifest
                )
            except KeyboardInterrupt:
                print(f"[STOPPED] Session {session_id} interrupted. Checkpoint saved to {DRIVE_PROCESSED_STREAMS_FILE}.")
                raise
            except Exception as exc:
                print(f"[ERROR] Session {session_id} failed: {exc}")
                success = False

            if success:
                processed_manifest[session_id] = {"completed": True, "last_frame_id": None}
                total_sessions_processed += 1

            if manifest_writable:
                save_processed_streams_manifest(processed_manifest)

        print(f"\n✅ Done! Total sessions fully processed: {total_sessions_processed}")
        print(f"✅ Processed sessions recorded on Drive: {len(processed_manifest)}")

No previous processed-stream manifest found on Drive. Starting fresh.
🚀 Starting pipeline for 20 unprocessed valid sessions (max 20). Output to /content/drive/My Drive/Capstone_Data_new/Data_new
  -> Starting Perception Pipeline on CUDA...
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=1, last_frame=0, speed=0.18 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=25, last_frame=72, speed=0.57 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=50, last_frame=147, speed=0.59 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=75, last_frame=222, speed=0.61 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=100, last_frame=297, speed=0.63 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=125, last_frame=372, speed=0.67 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=150, last_frame=450, speed=0.66 frames/s
[PROGRESS] -5OCPnbrwJdu3jH70ieU7pUiFsOJQoeG: frames=175, last_frame=525, speed=0.68 frames/s
[INFO] Stopping -5OCPnbr

# Clear the processed cache data

In [ ]:
import os
from pathlib import Path

print("🧨 Hard reset: clearing processed-session manifest and all SANPO_json outputs...")

drive_data_root = globals().get("DRIVE_DATA_ROOT")

if drive_data_root is None:
    print("[WARN] DRIVE_DATA_ROOT is not defined; rerun the Drive setup cell first.")
else:
    manifest_path = drive_data_root / "processed_streams.json"
    outputs_dir = drive_data_root / "Data_new"

    # 1) Delete processed-stream manifest
    try:
        if manifest_path.exists():
            manifest_path.unlink()
            print(f"⌒ Manifest deleted: {manifest_path}")
        else:
            print(f"[INFO] Manifest not found: {manifest_path}")
    except TimeoutError:
        print(f"[WARN] Drive timeout while deleting manifest: {manifest_path}")
    except Exception as exc:
        print(f"[WARN] Could not delete manifest '{manifest_path}': {exc}")

    # 2) Delete all session output files for a true hard reset
    deleted_outputs = 0
    try:
        if outputs_dir.exists():
            for jsonl_file in outputs_dir.glob("*.jsonl"):
                try:
                    jsonl_file.unlink()
                    deleted_outputs += 1
                except TimeoutError:
                    print(f"[WARN] Drive timeout while deleting output: {jsonl_file}")
                except Exception as exc:
                    print(f"[WARN] Could not delete output '{jsonl_file}': {exc}")
            print(f"⌒ Deleted {deleted_outputs} session output file(s) from {outputs_dir}")
        else:
            print(f"[INFO] Output folder not found: {outputs_dir}")
    except TimeoutError:
        print(f"[WARN] Drive timeout while scanning output folder: {outputs_dir}")
    except Exception as exc:
        print(f"[WARN] Could not scan output folder '{outputs_dir}': {exc}")

processed_cache = {}
print("Hard reset complete. Next run will reprocess from scratch unless new outputs are created.")

🧨 Hard reset: clearing processed-session manifest and all SANPO_json outputs...
⌒ Manifest deleted: /content/drive/My Drive/Capstone_Data_new/processed_streams.json
⌒ Deleted 11 session output file(s) from /content/drive/My Drive/Capstone_Data_new/Data_new
Hard reset complete. Next run will reprocess from scratch unless new outputs are created.


## View Output for One Session

In [ ]:
import json
from pathlib import Path

# Assuming processed_manifest contains the info of processed sessions
# Let's pick the last processed session from the manifest for demonstration
if processed_manifest:
    # Get the session ID of the last completed session
    last_session_id = list(processed_manifest.keys())[-1]
    session_output_path = DRIVE_SANPO_JSON_PATH / f"{last_session_id}.jsonl"

    print(f"Loading output for session: {last_session_id} from {session_output_path}")

    if session_output_path.exists():
        with open(session_output_path, 'r') as f:
            for line_num, line in enumerate(f):
                print(f"Frame {line_num+1}:")
                print(json.dumps(json.loads(line), indent=2))
                # Optionally, break after a few frames to avoid too much output
                if line_num >= 2: # Print first 3 frames
                    break
    else:
        print(f"Output file not found for session {last_session_id} at {session_output_path}")
else:
    print("No sessions have been processed yet or processed_manifest is empty.")

Loading output for session: 2iQ4jyIxwXUInJKDYmBuoDltgad_GKC5 from /content/drive/My Drive/Capstone_Data_new/Data_new/2iQ4jyIxwXUInJKDYmBuoDltgad_GKC5.jsonl
Frame 1:
{
  "schema_version": "1.0",
  "session_id": "2iQ4jyIxwXUInJKDYmBuoDltgad_GKC5",
  "frame_id": 0,
  "timestamp_s": 0.0,
  "fps": 15.0,
  "objects": []
}
Frame 2:
{
  "schema_version": "1.0",
  "session_id": "2iQ4jyIxwXUInJKDYmBuoDltgad_GKC5",
  "frame_id": 3,
  "timestamp_s": 0.2,
  "fps": 15.0,
  "objects": [
    {
      "object_id": "Object_01",
      "track_id": "5879",
      "class": "car",
      "distance_m": 6.58,
      "velocity_ms": 0.0,
      "bearing_deg": -13.85,
      "position_label": "left",
      "bbox_xyxy": [
        520,
        407,
        814,
        581
      ],
      "confidence": 0.75
    }
  ]
}
Frame 3:
{
  "schema_version": "1.0",
  "session_id": "2iQ4jyIxwXUInJKDYmBuoDltgad_GKC5",
  "frame_id": 6,
  "timestamp_s": 0.4,
  "fps": 15.0,
  "objects": [
    {
      "object_id": "Object_01",
      "tr